<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# Design Probes for the E6, E9 and E10 Plans

Probes, not tests. The plans in `docs/hypotheses/experiments/E6-E10.md` fix three small models before anything is
scored; E4's plan was parked because its network, written without a pilot, could not show what its hypothesis
changes. These probes check, on two or three seeds (1001 to 1003) that the plans' tests never use (their seeds are 1 to 20),
that each model is not decided by construction or degenerate - that the condition without the mechanism under test
does show the failure the mechanism is meant to prevent, and that the mechanism can change it. They informed the
choices each plan names; they decide nothing about any hunch. Three seeds give no interval, and none is claimed.

Run with `jupyter nbconvert --to notebook --execute --inplace notebooks/e6_e9_e10_design_probes.ipynb`; numpy as
pinned in `notebooks/requirements.txt`.

## E6: separation before completion

A store of M = 1000 units with k = 50 active (k-winners-take-all, ties to the lower index, synchronous steps, settled
when the active set repeats, at most 50 steps), written by the covariance rule. 50 patterns, either independent
(*random*) or in ten families of five whose members each keep `keep` of their family prototype's 50 units (*similar*).
With separation, every pattern and cue first passes through a fixed sparse random projection (connection probability
0.05, Gaussian weights) and k-winners-take-all. A cue moves 5 of a pattern's 50 active units. Success: the settled
state overlaps the target's stored code by at least 0.9 and more than any other stored code. Printed per run: success
share, unsettled share. The question for the plan: is there a family overlap at which the store without separation
interferes but not completely, so that the separation's effect is not fixed by a ceiling or a floor?

In [1]:
import numpy as np

M, K = 1000, 50
F = K / M

def kwta(h, k=K):
    s = np.zeros(h.shape[0])
    s[np.argsort(-h, kind="stable")[:k]] = 1.0
    return s

def patterns(rng, cond, keep, n=50, fam=5):
    out = []
    if cond == "random":
        for _ in range(n):
            s = np.zeros(M); s[rng.choice(M, K, replace=False)] = 1; out.append(s)
    else:
        for _ in range(n // fam):
            proto = rng.choice(M, K, replace=False)
            for _ in range(fam):
                kept = rng.choice(proto, keep, replace=False)
                add = rng.choice(np.setdiff1d(np.arange(M), kept), K - keep, replace=False)
                s = np.zeros(M); s[np.concatenate([kept, add])] = 1; out.append(s)
    return np.array(out)

def e6(seed, cond, sep, keep, moved=5, cues=100):
    rng = np.random.default_rng(seed)
    X = patterns(rng, cond, keep)
    R = rng.standard_normal((M, M)) * (rng.random((M, M)) < 0.05)
    enc = (lambda x: kwta(R @ x)) if sep else (lambda x: x)
    C = np.array([enc(x) for x in X])
    W = (C - F).T @ (C - F)
    np.fill_diagonal(W, 0)
    ok = unsettled = 0
    for _ in range(cues):
        mu = rng.integers(len(X)); x = X[mu].copy()
        x[rng.choice(np.flatnonzero(x), moved, replace=False)] = 0
        x[rng.choice(np.flatnonzero(X[mu] == 0), moved, replace=False)] = 1
        s = enc(x)
        for _ in range(50):
            s2 = kwta(W @ s)
            if np.array_equal(s2, s):
                break
            s = s2
        else:
            unsettled += 1
            continue
        ov = C @ s / K
        if ov[mu] >= 0.9 and (ov >= ov[mu]).sum() == 1:
            ok += 1
    return ok / cues, unsettled / cues

print("keep seed | random: off on | similar: off on   (success share, unsettled share)")
for keep in (20, 25, 35):
    for seed in (1001, 1002, 1003):
        r = {(c, s): e6(seed, c, s, keep) for c in ("random", "similar") for s in (0, 1)}
        print(keep, seed, "|", r[("random", 0)], r[("random", 1)], "|", r[("similar", 0)], r[("similar", 1)])


keep seed | random: off on | similar: off on   (success share, unsettled share)


20 1001 | (1.0, 0.0) (1.0, 0.0) | (1.0, 0.0) (1.0, 0.0)


20 1002 | (1.0, 0.0) (1.0, 0.0) | (1.0, 0.0) (1.0, 0.0)


20 1003 | (1.0, 0.0) (1.0, 0.0) | (1.0, 0.0) (1.0, 0.0)


25 1001 | (1.0, 0.0) (1.0, 0.0) | (0.42, 0.51) (1.0, 0.0)


25 1002 | (1.0, 0.0) (1.0, 0.0) | (0.52, 0.2) (0.96, 0.04)


25 1003 | (1.0, 0.0) (1.0, 0.0) | (0.32, 0.54) (0.96, 0.04)


35 1001 | (1.0, 0.0) (1.0, 0.0) | (0.0, 0.33) (0.04, 0.72)


35 1002 | (1.0, 0.0) (1.0, 0.0) | (0.0, 0.48) (0.05, 0.54)


35 1003 | (1.0, 0.0) (1.0, 0.0) | (0.0, 0.3) (0.21, 0.36)


## E9: a conserved economy of importance

32 units share 3200 integer tokens (100 each at the start) and a pool. Each tick each unit has input with probability
0.1 or 0.5, switching every 2000 ticks. The focus is every unit with input whose holding is at least a threshold, which
a controller moves by the focus's size minus 4 each tick (clamped at zero). Each focused unit is *used* with
probability 0.5. The economy charges each focused unit a rent of a twentieth of its holding (integer division); the
foil charges every unit with a token one token a tick, in focus or not. The pool is paid out equally to the used units
(integer division, remainder kept in the pool). Conservation is asserted every tick. Printed per run, over the second
half of 20 000 ticks: the largest share of all tokens any unit held; the share of ticks whose 100-tick moving average of
the focus's size is outside [3, 5]; the mean focus size. A first version of the foil, a levy of a 400th of each
holding, moved no token at all (every holding of 100 rounds it to zero), and was replaced by the flat levy below.

In [2]:
def e9(seed, rent, n=32, total=3200, target=4, ticks=20000, window=100):
    rng = np.random.default_rng(seed)
    hold = np.full(n, total // n, dtype=np.int64)
    pool = theta = 0
    largest = 0.0
    sizes = []
    for t in range(ticks):
        p = 0.1 if (t // 2000) % 2 == 0 else 0.5
        focus = np.flatnonzero((rng.random(n) < p) & (hold >= theta))
        used = focus[rng.random(focus.size) < 0.5]
        charge = hold[focus] // 20 if rent else (hold > 0).astype(np.int64)
        if rent:
            hold[focus] -= charge
        else:
            hold -= charge
        pool += int(charge.sum())
        if used.size:
            share = pool // used.size
            hold[used] += share
            pool -= share * used.size
        theta = max(0, theta + focus.size - target)
        assert hold.sum() + pool == total and (hold >= 0).all()
        sizes.append(focus.size)
        if t >= ticks // 2:
            largest = max(largest, hold.max() / total)
    s = np.array(sizes[ticks // 2:])
    avg = np.convolve(s, np.ones(window) / window, "valid")
    return round(float(largest), 3), round(float(np.mean((avg < target - 1) | (avg > target + 1))), 3), round(float(s.mean()), 2)

print("seed | rent: largest share, outside band, mean focus | flat levy: the same")
for seed in (1001, 1002, 1003):
    print(seed, "|", e9(seed, True), "|", e9(seed, False))


seed | rent: largest share, outside band, mean focus | flat levy: the same


1001 | (0.089, 0.043, 3.7) | (0.279, 0.193, 3.6)


1002 | (0.096, 0.035, 3.7) | (0.31, 0.207, 3.6)


1003 | (0.092, 0.054, 3.71) | (0.298, 0.186, 3.62)


## E10: input that reshapes the landscape

A Hopfield network of N = 200 sign units holding P = 10 random patterns by Hebb's rule (zero diagonal), updated one unit
at a time in index order (a unit keeps its state at zero input), settled when a sweep changes nothing, at most 50
sweeps. A cue takes each unit from pattern a with probability 0.6 and from pattern b otherwise (*blended*); in the
*shifting* condition the cue is redrawn before each of the first ten sweeps with that probability falling from 0.8 by
0.06 a sweep, and held after. The cue is also a persistent field of strength gamma. The baseline weighs every pattern
alike; the modulated network weighs pattern mu by the square of its positive overlap with the current cue,
recomputed each sweep, either as is or normalised so the weights sum to P. Printed per run: the share of cues settling
clean (signed overlap at least 0.9 with one pattern), unsettled, and on a mirror (overlap at most -0.9).

In [3]:
N = 200

def e10(seed, gamma, mode, shift, P=10, cues=100):
    rng = np.random.default_rng(seed)
    X = rng.choice([-1.0, 1.0], size=(P, N))
    W0 = X.T @ X / N
    np.fill_diagonal(W0, 0)
    clean = unsettled = mirror = 0
    for _ in range(cues):
        a, b = rng.choice(P, 2, replace=False)
        u = np.where(rng.random(N) < 0.6, X[a], X[b])
        s = u.copy()
        settled = False
        for sweep in range(50):
            if shift and sweep < 10:
                u = np.where(rng.random(N) < 0.8 - 0.06 * sweep, X[a], X[b])
            if mode == "baseline":
                W = W0
            else:
                w = np.maximum(X @ u / N, 0) ** 2
                if mode == "normalised":
                    w = w * P / w.sum() if w.sum() > 0 else np.ones(P)
                W = (X.T * w) @ X / N
                np.fill_diagonal(W, 0)
            changed = 0
            for i in range(N):
                h = W[i] @ s + gamma * u[i]
                new = s[i] if h == 0 else (1.0 if h > 0 else -1.0)
                if new != s[i]:
                    s[i] = new; changed += 1
            if changed == 0 and not (shift and sweep < 10):
                settled = True
                break
        if not settled:
            unsettled += 1
            continue
        ov = X @ s / N
        if ov.max() >= 0.9:
            clean += 1
        elif ov.min() <= -0.9:
            mirror += 1
    return clean / cues, unsettled / cues, mirror / cues

print("condition gamma seed | baseline | unnormalised | normalised   (clean, unsettled, mirror)")
for shift in (False, True):
    for gamma in (0.0, 0.5):
        for seed in (1001, 1002):
            print("shifting" if shift else "blended", gamma, seed, "|",
                  e10(seed, gamma, "baseline", shift), "|", e10(seed, gamma, "unnormalised", shift), "|",
                  e10(seed, gamma, "normalised", shift))


condition gamma seed | baseline | unnormalised | normalised   (clean, unsettled, mirror)


blended 0.0 1001 | (0.94, 0.0, 0.01) | (1.0, 0.0, 0.0) | (1.0, 0.0, 0.0)


blended 0.0 1002 | (0.87, 0.0, 0.0) | (1.0, 0.0, 0.0) | (1.0, 0.0, 0.0)


blended 0.5 1001 | (0.4, 0.0, 0.0) | (0.0, 0.0, 0.0) | (0.99, 0.0, 0.0)


blended 0.5 1002 | (0.31, 0.0, 0.0) | (0.0, 0.0, 0.0) | (0.97, 0.0, 0.0)


shifting 0.0 1001 | (0.91, 0.0, 0.0) | (0.81, 0.0, 0.19) | (0.81, 0.0, 0.19)


shifting 0.0 1002 | (0.79, 0.0, 0.02) | (0.85, 0.0, 0.15) | (0.85, 0.0, 0.15)


shifting 0.5 1001 | (0.98, 0.0, 0.0) | (0.12, 0.0, 0.0) | (0.92, 0.0, 0.08)


shifting 0.5 1002 | (0.86, 0.0, 0.0) | (0.17, 0.0, 0.0) | (0.95, 0.0, 0.05)


## What the probes informed

Read with the printed tables above, on these seeds only:

- **E6.** The family overlap is fixed at `keep` = 25, where the store without separation fails on part of the similar
  cues but not on all; at 20 neither variant fails, and at 35 both fail on most. Unsettled cues are frequent in the
  similar condition without separation, so the plan counts them as failures to retrieve there and applies its void
  rule only to the random condition.
- **E9.** The foil is the flat levy, since the proportional one moved nothing; the controller's band and the 0.05
  margin of the plan were written before this probe ran.
- **E10.** The modulation is normalised to sum to P: unnormalised, it shrinks the recurrent drive against the field
  and loses to the baseline whenever the field is on. The field is fixed at gamma = 0.5, where the baseline smears a
  blended cue on most cues and at gamma = 0 on few. In the shifting condition the baseline settled clean on most cues
  at both values of gamma, so that condition may meet the plan's ceiling gate; and there the normalised modulation
  settled on a mirror of a pattern on some cues, which the plan counts as not clean.